# Apparel and footwear: 2025 trade-policy shocks

**What this notebook does.** It reads US imports of apparel and footwear from `core` — the ten
largest HS-6 codes by tonnage across chapters 61 (knit apparel), 62 (woven apparel) and 64
(footwear) — and shows where supply moved between origins and modes around the 2025
trade-policy shocks, and what that did to the transport emissions of delivering a tonne to
the US.

**Why apparel.** Data center hardware is a weak test for both shocks: servers were exempt from
the reciprocal tariffs, and the electrical gear's import volumes are small. Apparel is the
proposed lead product: high tonnage, diverse origins, clear tariff exposure, and China's share
is the observable response. Vietnam, Bangladesh, Indonesia and Cambodia are the main
alternatives.

**The results at a glance** are in Section 8.

**How it's organised.** Each section computes with the engine on data read by
`analysis/src/shockt_analysis/core.py`. The windows reuse the Case 1 tariff timeline.

**Data gaps.** Bangladesh and Cambodia have no container-ship voyage data in Climate TRACE, so
their vessel lane emissions cannot be computed. Indonesia's coverage is sparse (1 departure
port, ~10 of 55 months). Together these origins account for about 13% of apparel import value.

**To run:** set `ANALYST_DATABASE_URL` in `analysis/.env`, then run all cells.

## 1. Setup

The product group follows `config/hs_codes.yaml` under `apparel_footwear`. The windows are
the Case 1 tariff timeline (the reciprocal tariffs apply to apparel the same as to servers).
Named origins are the top six by import value.

In [ ]:
from datetime import date

import matplotlib.pyplot as plt
import polars as pl
from shockt_engine import (
    Window,
    load_leg_factors,
    mix_decomposition,
    select_factors,
    shares,
    window_summary,
)

from shockt_analysis import cases, core
from shockt_analysis.settings import REPO_ROOT, get_settings

pl.Config.set_tbl_rows(40)
pl.Config.set_tbl_width_chars(160)
pl.Config.set_float_precision(3)

CASE = "tariffs_2025"
CASE_DIR = REPO_ROOT / "inputs" / "cases" / CASE
NOTEBOOK = "analysis/apparel/apparel_tariffs_2025.ipynb"

APPAREL = [
    "611020", "610910", "611030", "640299", "620462",
    "640419", "620342", "640399", "640391", "640411",
]
NAMED_ORIGINS = ["VNM", "CHN", "BGD", "IDN", "KHM", "IND"]

windows = cases.load_windows(CASE_DIR / "shock_windows.yaml")
WINDOW_COLORS = {"baseline": "#d9e5d6", "anticipation": "#f3dfb6", "response": "#f0c7c0"}
[(w.name, w.first, w.last) for w in windows]

Read everything the notebook needs in one connection.

In [ ]:
with core.connect(get_settings()) as conn:
    supply = core.supply(conn, APPAREL)
    air = core.air_values(conn, APPAREL)
    node_weights = core.node_weights(conn, APPAREL)
    observed_sea = core.observed_sea(conn)
    distances = {seed: core.distances(conn, seed) for seed in ("sea", "air", "road", "rail")}
    crossing_plants = core.crossing_plants(conn)
    factor_rows = core.factors(conn)
    loads = core.current_loads(
        conn,
        [
            "census_hs",
            "census_porths",
            "census_statehs",
            "bts_transborder",
            "climatetrace_voyages",
            "bts_t100_segments",
            "uk_ghg_conversion_factors",
            "epa_ghg_transport_factors",
            "epa_ghg_gwp",
        ],
    )

leg_factors = select_factors(factor_rows, load_leg_factors())
context = cases.LaneContext(
    weights=node_weights,
    observed_sea=observed_sea,
    distances=distances,
    factors={mode: f.wtw_kg_co2e_per_tkm for mode, f in leg_factors.items()},
    crossing_plants=crossing_plants,
)
print(f"Supply rows: {supply.height:,}; months {supply['month'].min()} to {supply['month'].max()}")
pl.DataFrame(
    [
        {
            "mode": mode,
            "factor": f.variant,
            "source": f.source,
            "kg CO2e per tonne-km": f.wtw_kg_co2e_per_tkm,
        }
        for mode, f in leg_factors.items()
    ]
)

In [ ]:
loads

## 2. Monthly supply

Tonnes of apparel and footwear delivered to the US each month, by origin, with the windows
shaded. Air and vessel weights are reported by Census; Mexico's truck and rail are estimated.

In [ ]:
def shade_windows(ax):
    for w in windows:
        ax.axvspan(
            w.first,
            date(w.last.year + (w.last.month == 12), w.last.month % 12 + 1, 1),
            color=WINDOW_COLORS[w.name],
            alpha=0.5,
            lw=0,
            label=w.name,
        )


def plot_by_origin(codes, named, title):
    monthly = cases.monthly_supply(supply, codes, named)
    pivot = (
        monthly.group_by("month", "origin_group")
        .agg(pl.col("weight_t").sum())
        .pivot(on="origin_group", index="month", values="weight_t")
        .sort("month")
        .fill_null(0)
    )
    order = [o for o in named + [cases.OTHER] if o in pivot.columns]
    fig, ax = plt.subplots(figsize=(11, 4))
    shade_windows(ax)
    ax.stackplot(pivot["month"].to_list(), *[pivot[o].to_list() for o in order], labels=order)
    ax.set_title(title)
    ax.set_ylabel("tonnes per month")
    ax.legend(loc="upper left", ncols=4, fontsize=8)
    plt.show()
    return monthly


apparel_monthly = plot_by_origin(
    APPAREL, NAMED_ORIGINS, "Apparel and footwear delivered to the US, by origin"
)

Mode shares within each origin. Apparel is overwhelmingly vessel (~96%), unlike servers which
are majority air from several origins.

In [ ]:
mode_shares = cases.window_shares(apparel_monthly, windows, ["origin_group", "mode"])
by_origin = mode_shares.group_by("window", "origin_group").agg(
    pl.col("weight_t").sum().alias("origin_t")
)
(
    mode_shares.join(by_origin, on=["window", "origin_group"])
    .with_columns((pl.col("weight_t") / pl.col("origin_t")).alias("share_within_origin"))
    .pivot(on="window", index=["origin_group", "mode"], values="share_within_origin")
    .select("origin_group", "mode", *[w.name for w in windows])
    .sort("origin_group", "mode")
)

## 3. Origin shares and changes

Each origin's share of apparel tonnage per window, and the change from baseline to response.
The meeting notes expect China's share to fall and Vietnam and Mexico to gain.

In [ ]:
origin_shares = cases.window_shares(apparel_monthly, windows, ["origin_group"])
shares_table = origin_shares.pivot(
    on="window", index="origin_group", values="share"
).select("origin_group", *[w.name for w in windows])
changes = cases.share_changes(origin_shares, "baseline", "response", "origin_group")
shares_table.with_columns(
    pl.col("origin_group")
    .replace_strict(changes, default=None)
    .alias("change_baseline_to_response")
)

The same by value instead of tonnage, as a robustness check. If value-based shares move
differently from tonnage-based shares, the estimated weights may be driving the finding.

In [ ]:
value_monthly = (
    cases.with_origin_group(
        supply.filter(pl.col("hs6").is_in(APPAREL)),
        NAMED_ORIGINS,
    )
)
value_by_window = (
    cases.with_window(value_monthly, windows)
    .drop_nulls("window")
    .group_by("window", "origin_group")
    .agg(pl.col("weight_t").sum())
)

# Also compute value-based shares from fct_imports_monthly value_usd
# supply() doesn't carry value, so read it from the raw import data via an inline query
with core.connect(get_settings()) as conn:
    value_supply = core.frame(
        conn,
        """
        select i.month, c.iso3, sum(i.value_usd)::float8 as value_usd
        from core.fct_imports_monthly as i
        join core.dim_country as c using (census_cty_code)
        where i.hs6 = any(%(hs6)s) and c.region <> 'other'
        group by i.month, c.iso3
        """,
        {"hs6": APPAREL},
    )

value_with_groups = cases.with_origin_group(value_supply, NAMED_ORIGINS)
value_windowed = (
    cases.with_window(value_with_groups, windows)
    .drop_nulls("window")
    .group_by("window", "origin_group")
    .agg(pl.col("value_usd").sum())
)
value_shares_rows = []
for window_name in [w.name for w in windows]:
    part = value_windowed.filter(pl.col("window") == window_name)
    if part.is_empty():
        continue
    s = shares(dict(zip(
        part["origin_group"].to_list(),
        part["value_usd"].to_list(),
        strict=True,
    )))
    for origin, share in s.items():
        value_shares_rows.append({
            "window": window_name,
            "origin_group": origin,
            "value_share": share,
        })

value_shares_df = pl.DataFrame(value_shares_rows)
value_table = value_shares_df.pivot(
    on="window", index="origin_group", values="value_share"
).select("origin_group", *[w.name for w in windows])

print("Origin shares by VALUE (USD):")
display(value_table.sort("origin_group"))
print("\nOrigin shares by TONNAGE (for comparison):")
display(shares_table.sort("origin_group"))

## 4. By HS-6 code

Apparel is ten codes across three chapters. Do all ten move alike, or does one product drive
the aggregate? The top codes by tonnage and their origin-share changes.

In [ ]:
code_monthly = (
    cases.with_window(supply.filter(pl.col("hs6").is_in(APPAREL)), windows)
    .drop_nulls("window")
    .group_by("window", "hs6")
    .agg(pl.col("weight_t").sum())
)
code_shares = []
for window_name in [w.name for w in windows]:
    part = code_monthly.filter(pl.col("window") == window_name)
    s = shares(dict(zip(part["hs6"].to_list(), part["weight_t"].to_list(), strict=True)))
    for code, share in s.items():
        code_shares.append({"window": window_name, "hs6": code, "share": share})

code_shares_df = pl.DataFrame(code_shares)
code_shares_df.pivot(on="window", index="hs6", values="share").select(
    "hs6", *[w.name for w in windows]
).sort("baseline", descending=True)

China's share change by code. A consistent drop across codes strengthens the finding; one
code driving the aggregate weakens it.

In [ ]:
china_by_code = []
for hs6 in APPAREL:
    hs_supply = supply.filter(pl.col("hs6") == hs6)
    hs_monthly = cases.monthly_supply(hs_supply, [hs6], NAMED_ORIGINS)
    ws = cases.window_shares(hs_monthly, windows, ["origin_group"])
    ch = cases.share_changes(ws, "baseline", "response", "origin_group")
    baseline_row = ws.filter(
        (pl.col("window") == "baseline") & (pl.col("origin_group") == "CHN")
    )
    baseline_share = baseline_row["share"].item() if not baseline_row.is_empty() else 0
    china_by_code.append({
        "hs6": hs6,
        "china_baseline_share": baseline_share,
        "china_change_pp": ch.get("CHN", 0) * 100,
    })

china_df = pl.DataFrame(china_by_code).sort("china_change_pp")
fig, ax = plt.subplots(figsize=(9, 4))
ax.barh(
    china_df["hs6"].to_list(),
    china_df["china_change_pp"].to_list(),
    color=["#a2412d" if v < 0 else "#2d5b4f" for v in china_df["china_change_pp"].to_list()],
)
ax.set_xlabel("China share change, percentage points")
ax.set_title("China's share change by HS-6 code, baseline to response")
ax.axvline(0, color="black", lw=0.5)
plt.tight_layout()
plt.show()
china_df

## 5. Delivery emissions per tonne, by lane

A lane is an origin and a delivery mode. Its emissions per tonne are the sum over its legs of
distance × factor, with each leg's distance averaged over the airports, ports and states the
lane uses. Sea distances are observed where they exist, and `searoute` distances otherwise.

**Known gap:** Bangladesh and Cambodia have no container-ship voyage data, so their vessel
lanes will show null intensity.

In [ ]:
apparel_lanes, apparel_legs = cases.lane_intensities(supply, APPAREL, windows, context)
(
    apparel_lanes.filter(pl.col("iso3").is_in(NAMED_ORIGINS))
    .pivot(on="window", index=["iso3", "mode"], values="kg_co2e_per_t")
    .select("iso3", "mode", *[w.name for w in windows])
    .sort("iso3", "mode")
)

In [ ]:
print("Lanes with tonnage but no measurable legs (left out of the mix):")
missing = apparel_lanes.filter(pl.col("kg_co2e_per_t").is_null())
display(missing)
missing_t = missing["weight_t"].sum()
total_t = apparel_lanes["weight_t"].sum()
print(
    f"\n{missing_t:,.0f} t of {total_t:,.0f} t total ({missing_t / total_t:.1%}) have no "
    f"measurable legs."
)

## 6. The supply mix

The emissions per tonne of all US apparel supply in a month: each origin lane's share of the
month's tonnes × the lane's intensity. This is the case's headline, and it doesn't move with
total volume, only with where and how supply arrives.

In [ ]:
mix = {
    estimate: cases.mix_by_month(
        supply, APPAREL, windows, apparel_lanes, tonnes=column
    )
    for estimate, column in [
        ("central", "weight_t"),
        ("mexico low", "weight_t_low"),
        ("mexico high", "weight_t_high"),
    ]
}
fig, ax = plt.subplots(figsize=(11, 4))
shade_windows(ax)
central = mix["central"]
ax.plot(
    central["month"].to_list(),
    central["kg_co2e_per_t"].to_list(),
    marker="o",
    color="#2d5b4f",
    label="central",
)
ax.fill_between(
    central["month"].to_list(),
    mix["mexico high"]["kg_co2e_per_t"].to_list(),
    mix["mexico low"]["kg_co2e_per_t"].to_list(),
    color="#2d5b4f",
    alpha=0.15,
    label="Mexico tonnage range",
)
ax.set_ylabel("kg CO2e per tonne delivered")
ax.set_title("Emissions per tonne of US apparel supply")
ax.legend(loc="upper left", fontsize=8)
plt.show()

values = dict(zip(central["month"].to_list(), central["kg_co2e_per_t"].to_list(), strict=True))
weights = dict(zip(central["month"].to_list(), central["t"].to_list(), strict=True))
mix_summary = window_summary(values, windows, weights)
mix_table = pl.DataFrame(
    [
        {
            "window": name,
            "mean_kg_per_t": s.mean,
            "low": s.low,
            "high": s.high,
            "months": s.months,
            "change_from_baseline": s.mean / mix_summary["baseline"].mean - 1
            if name != "baseline"
            else 0.0,
        }
        for name, s in mix_summary.items()
    ]
)
mix_table

In [ ]:
print(f"Measured share of tonnage (central): "
      f"{central['measured_share'].mean():.1%} of monthly supply has a measured lane.")

## 7. What moved the mix

The Shapley decomposition: was the change in emissions per tonne driven by the origin mix
(which countries supply how much), the mode within each origin (air vs vessel), or the route
legs (sea distance, inland distribution)?

In [ ]:
apparel_cells = {
    w.name: cases.lane_mix_cells(supply, APPAREL, w, apparel_legs) for w in windows
}
apparel_split = mix_decomposition(
    apparel_cells["baseline"], apparel_cells["response"], ["origin", "mode"]
)
fig, ax = plt.subplots(figsize=(9, 4))
cases.plot_waterfall(
    ax,
    apparel_split,
    {
        "origin": "origin mix",
        "mode": "mode within origin",
        "international": "legs to the US",
        "inland": "legs in the US",
    },
    "What moved the emissions per tonne of US apparel supply, baseline to response",
    window_label="response",
)
plt.show()
change = apparel_split.window - apparel_split.baseline
if abs(change) > 0:
    print(
        f"Of the {change:+,.0f} kg CO2e per tonne change, the origin mix accounts for "
        f"{apparel_split.effects['origin']:+,.0f} kg "
        f"({apparel_split.effects['origin'] / change:.0%}), "
        f"the mode within each origin {apparel_split.effects['mode']:+,.0f}, the legs to the US "
        f"{apparel_split.effects['international']:+,.0f} and the legs inside it "
        f"{apparel_split.effects['inland']:+,.0f}."
    )
else:
    print("No measurable change between baseline and response.")

## 8. Results at a glance

Three charts: **emissions per tonne** of US apparel supply by period, **tonnes per month by
origin**, and **what moved the mix**. The periods are the case's windows plus the months after
to the last loaded month.

In [ ]:
last_window = windows[-1].last
after = Window(
    "after",
    date(last_window.year + last_window.month // 12, last_window.month % 12 + 1, 1),
    supply["month"].max(),
)
after_lanes, _ = cases.lane_intensities(supply, APPAREL, [after], context)
periods = [*windows, after]
period_lanes = pl.concat([apparel_lanes, after_lanes])

# Chart 1: Emissions per tonne by period
monthly_mix = cases.mix_by_month(supply, APPAREL, periods, period_lanes)
months = monthly_mix["month"].to_list()
summary = window_summary(
    dict(zip(months, monthly_mix["kg_co2e_per_t"].to_list(), strict=True)),
    periods,
    dict(zip(months, monthly_mix["t"].to_list(), strict=True)),
)
period_labels = [
    f"{w.first:%b %Y}\u2013{w.last:%b %Y}\n({w.name})" for w in windows
] + [f"{after.first:%b %Y}\u2013{after.last:%b %Y}\n(after)"]
means = [summary[w.name].mean for w in periods]

fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(
    period_labels,
    means,
    color=[WINDOW_COLORS.get(w.name, "#e0e0e0") for w in periods],
    edgecolor="#2d5b4f",
)
ax.set_ylabel("kg CO2e per tonne delivered")
ax.set_title("US apparel supply: emissions per tonne by period")
ax.tick_params(axis="x", labelsize=8)
plt.show()

base_mean = summary["baseline"].mean
resp_mean = summary["response"].mean
print(
    f"Emissions per tonne: {base_mean:.1f} kg in baseline, {resp_mean:.1f} kg in response "
    f"({resp_mean / base_mean - 1:+.0%})."
)

In [ ]:
# Chart 2: Tonnes per month by origin
origin_tonnes = (
    cases.with_window(apparel_monthly, windows)
    .drop_nulls("window")
    .group_by("window", "origin_group")
    .agg(pl.col("weight_t").sum())
    .with_columns(
        (
            pl.col("weight_t")
            / pl.col("window").replace_strict(
                {
                    w.name: (w.last.year - w.first.year) * 12 + w.last.month - w.first.month + 1
                    for w in windows
                }
            )
        ).alias("t_per_month")
    )
    .pivot(on="window", index="origin_group", values="t_per_month")
    .with_columns((pl.col("response") / pl.col("baseline")).alias("growth"))
    .sort("baseline", descending=True)
)

fig, ax = plt.subplots(figsize=(9, 4))
groups_ = origin_tonnes["origin_group"].to_list()
x = range(len(groups_))
ax.bar(
    [i - 0.2 for i in x],
    origin_tonnes["baseline"].to_list(),
    0.4,
    label="baseline",
    color=WINDOW_COLORS["baseline"],
    edgecolor="#2d5b4f",
)
ax.bar(
    [i + 0.2 for i in x],
    origin_tonnes["response"].to_list(),
    0.4,
    label="response",
    color=WINDOW_COLORS["response"],
    edgecolor="#a2412d",
)
for i, g in zip(x, origin_tonnes["growth"].to_list(), strict=True):
    if g is not None:
        ax.annotate(
            f"\u00d7{g:.2f}",
            (i + 0.2, origin_tonnes["response"][i]),
            ha="center",
            va="bottom",
            fontsize=8,
        )
ax.set_xticks(list(x), groups_)
ax.set_ylabel("tonnes per month")
ax.set_title("Apparel delivered to the US: tonnes per month by origin")
ax.legend(fontsize=8)
plt.show()

## 9. Findings

The DERIVED statements below are computed from the cells above.

In [ ]:
chn_change = changes.get("CHN", 0) * 100
vnm_change = changes.get("VNM", 0) * 100
bgd_change = changes.get("BGD", 0) * 100
idn_change = changes.get("IDN", 0) * 100
mex_change = changes.get("MEX", 0) if "MEX" in changes else None

base, resp = mix_summary["baseline"], mix_summary["response"]

# Count missing lanes
missing_lanes = apparel_lanes.filter(pl.col("kg_co2e_per_t").is_null())
missing_origins = sorted(missing_lanes["iso3"].unique().to_list())

findings = [
    f"China's share of US apparel tonnage changed by {chn_change:+.1f} percentage points from "
    f"the baseline to the response window.",
    f"Vietnam's share changed by {vnm_change:+.1f} pp, Bangladesh by {bgd_change:+.1f} pp, "
    f"Indonesia by {idn_change:+.1f} pp.",
    f"The emissions per tonne of US apparel supply averaged {base.mean:.1f} kg CO2e in the "
    f"baseline and {resp.mean:.1f} kg in the response window "
    f"({resp.mean / base.mean - 1:+.0%}).",
]

if abs(change) > 0:
    findings.append(
        f"The origin mix accounts for {apparel_split.effects['origin']:+,.0f} kg of the "
        f"{change:+,.0f} kg change; the mode mix {apparel_split.effects['mode']:+,.0f} kg; "
        f"legs to the US {apparel_split.effects['international']:+,.0f} kg; "
        f"legs inside the US {apparel_split.effects['inland']:+,.0f} kg."
    )

if missing_origins:
    findings.append(
        f"Vessel lanes for {', '.join(missing_origins)} have no measurable emissions (no "
        f"Climate TRACE voyage data); {missing_lanes['weight_t'].sum():,.0f} t of "
        f"{apparel_lanes['weight_t'].sum():,.0f} t total "
        f"({missing_lanes['weight_t'].sum() / apparel_lanes['weight_t'].sum():.1%}) are excluded "
        f"from the mix."
    )

for i, f in enumerate(findings, 1):
    print(f"{i}. {f}")

**What the data suggests (INFERENCE, for the team to review).** Unlike servers — which were
exempt from the reciprocal tariffs and whose shift toward Taiwan by air is better explained by
AI demand growth — apparel was fully exposed. China is the observable origin: if its share
fell while Vietnam, Bangladesh and Indonesia gained, that is consistent with tariff-induced
diversification. Apparel is overwhelmingly shipped by vessel, so the emissions per tonne are
set by ocean distance, not by an air-vs-vessel mode shift. A shift from China (shorter Pacific
route) to Vietnam or Bangladesh (longer routes, some through the Suez) would raise emissions
per tonne modestly. The data gaps — no voyage data for Bangladesh and Cambodia — mean the mix
figure excludes about 13% of tonnage; filling that gap would tighten the headline number.

**What it can't say.** Monthly data can't separate the tariff announcements from seasonal or
sourcing changes in the same months. The independence assumption for destination states applies
here too. Apparel air weights are a small share but may be imputed by Census in the same way
as server air weights.